# 🤖 AI Agents — Hands-On Lab## Build agents that THINK, ACT, OBSERVE, and REPEAT.**What this notebook does:**- Build a ReAct agent from scratch (no frameworks)- Implement tool/function calling with OpenAI- Build a LangGraph state machine with conditional routing- Multi-agent team (researcher + writer)- Memory systems (short-term + entity extraction)- Human-in-the-loop approval gates- Code execution in a sandbox**Setup:**```pip install openai langchain langchain-openai langgraph```**Cost:** ~$0.50 - $1.00 to run everything

In [ ]:
# ============================================================
# SETUP — Run this first!
# ============================================================

# pip install openai langchain langchain-openai langgraph
from openai import OpenAI
import json, time, re

client = OpenAI()

def ask(prompt, model="gpt-4o-mini", temperature=0, system=None):
    messages = []
    if system: messages.append({"role":"system","content":system})
    messages.append({"role":"user","content":prompt})
    r = client.chat.completions.create(model=model, messages=messages, temperature=temperature)
    return r.choices[0].message.content.strip(), r.usage.total_tokens

answer, tokens = ask("Say 'Agent ready' in 2 words.")
print(f"✅ Setup complete! {answer}")

---# 1️⃣ ReAct Agent (From Scratch — No Frameworks)**Analogy:** A detective solving a case. THINK: "I should check the security cameras." ACT: Goes and checks cameras. OBSERVE: "Camera shows the suspect at 9pm." THINK again. Repeat until solved.**What we build:** A working ReAct agent with 3 tools, built from pure Python. No LangChain needed.

In [ ]:
# ============================================================
# REACT AGENT: Think -> Act -> Observe -> Repeat
# Built from scratch so you see EXACTLY how it works
# ============================================================

# ---------- TOOLS (the agent's "hands") ----------

def search_database(query):
    """Search our company knowledge base for policies and info."""
    db = {
        "refund": "Refund policy: 30 days for standard items. Electronics: 15 days.",
        "pricing": "Pro plan: $49/month. Enterprise: $199/month. Annual: 20% off.",
        "warranty": "1-year warranty for defects. Physical damage NOT covered.",
        "shipping": "Free over $50. Standard: 5-7 days. Express: $15 for 2-day.",
    }
    query_lower = query.lower()
    for key, value in db.items():
        if key in query_lower:
            return value
    return "No results found for that query."

def calculator(expression):
    """Calculate a math expression. Only use AFTER getting numbers from search."""
    try:
        # Safe eval for math only
        allowed = set("0123456789+-*/.() ")
        if all(c in allowed for c in expression):
            return str(eval(expression))
        return "Error: invalid expression"
    except:
        return "Error: could not calculate"

def get_current_date():
    """Get today's date."""
    from datetime import date
    return str(date.today())

# Registry of available tools
TOOLS = {
    "search_database": {
        "func": search_database,
        "description": "Search company knowledge base. Use for policy/pricing questions."
    },
    "calculator": {
        "func": calculator,
        "description": "Calculate math. Use AFTER getting numbers from search."
    },
    "get_current_date": {
        "func": get_current_date,
        "description": "Get today's date. Use when question involves dates."
    },
}


# ---------- THE REACT LOOP ----------

def react_agent(question, max_steps=5):
    """
    The ReAct loop:
    1. THINK about what to do next
    2. ACT by calling a tool
    3. OBSERVE the result
    4. REPEAT until we have the answer
    """
    
    # Build tool descriptions for the LLM
    tool_list = "\n".join([
        f"  - {name}: {info['description']}" 
        for name, info in TOOLS.items()
    ])
    
    # Conversation history (the agent's "memory" during this task)
    history = []
    
    print(f"  Question: {question}")
    print(f"  {'─' * 55}")
    
    for step in range(1, max_steps + 1):
        
        # Ask the LLM to think and decide the next action
        prompt = f"""You are a helpful agent that answers questions using tools.

Available tools:
{tool_list}

Question: {question}

Previous steps:
{chr(10).join(history) if history else "None yet."}

What should you do next? Choose ONE:
A) Use a tool: respond with EXACTLY this format:
   THINK: [your reasoning]
   ACTION: [tool_name]
   INPUT: [input for the tool]

B) Give the final answer: respond with EXACTLY:
   THINK: [your reasoning]
   ANSWER: [your final answer]"""
        
        response, tokens = ask(prompt)
        
        # Parse the response
        if "ANSWER:" in response:
            # Agent is ready to answer
            think = response.split("THINK:")[-1].split("ANSWER:")[0].strip()
            answer = response.split("ANSWER:")[-1].strip()
            print(f"  Step {step} THINK: {think[:60]}...")
            print(f"  Step {step} ✅ ANSWER: {answer}")
            return answer
        
        elif "ACTION:" in response:
            # Agent wants to use a tool
            think = response.split("THINK:")[-1].split("ACTION:")[0].strip()
            action = response.split("ACTION:")[-1].split("INPUT:")[0].strip()
            tool_input = response.split("INPUT:")[-1].strip()
            
            print(f"  Step {step} THINK: {think[:60]}...")
            print(f"  Step {step} ACT:   {action}('{tool_input}')")
            
            # Execute the tool
            if action in TOOLS:
                result = TOOLS[action]["func"](tool_input)
                print(f"  Step {step} SEE:   {result[:60]}...")
                history.append(f"Step {step}: Used {action}('{tool_input}') → {result}")
            else:
                result = f"Unknown tool: {action}"
                history.append(f"Step {step}: Error - {result}")
                print(f"  Step {step} ERROR: {result}")
        else:
            print(f"  Step {step} (unparsed): {response[:60]}...")
            history.append(f"Step {step}: {response[:100]}")
    
    return "Could not find answer within step limit."


# ---------- TEST ----------

print("REACT AGENT (built from scratch)")
print("=" * 60)
print()

test_questions = [
    "What is the Pro plan price for a full year with annual discount?",
    "Can I return a laptop I bought 20 days ago?",
    "What is today's date?",
]

for q in test_questions:
    react_agent(q)
    print()

print("💡 The agent THINKS before each action. Like a detective following clues.")
print("💡 Each step: reason → pick tool → use it → see result → reason again.")
print("💡 Max 5 steps prevents infinite loops.")

---# 2️⃣ Tool / Function Calling (OpenAI Native)**Analogy:** Your smart assistant has no hands. They DECIDE which tool to use and what to do with it. But YOUR CODE actually does the work. The LLM thinks, you execute. This is the safety boundary.**What we build:** Define 3 tools. LLM picks which one to call (including parallel calls). We execute safely.

In [ ]:
# ============================================================
# FUNCTION CALLING: LLM decides WHICH tool. YOUR code executes.
# This is HOW production agents work (not string parsing!)
# ============================================================

# ---------- DEFINE TOOLS (tell the LLM what's available) ----------

tools = [
    {
        "type": "function",
        "function": {
            "name": "search_knowledge_base",
            "description": "Search company knowledge base for product info, policies, and pricing. Use this for any factual question about our products.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "What to search for, e.g. 'refund policy' or 'pricing'"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform math calculations. Use AFTER getting numbers from search.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Math expression like '49 * 12 * 0.8'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "Send an email to a customer. REQUIRES APPROVAL before sending.",
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {"type": "string", "description": "Email address"},
                    "subject": {"type": "string", "description": "Email subject"},
                    "body": {"type": "string", "description": "Email body text"}
                },
                "required": ["to", "subject", "body"]
            }
        }
    },
]

# ---------- TOOL IMPLEMENTATIONS (your code, not the LLM's) ----------

def execute_tool(name, arguments):
    """YOUR code runs the tool. The LLM only DECIDED which tool to use."""
    
    if name == "search_knowledge_base":
        # Same mock database
        db = {"refund": "30 days standard. Electronics: 15 days.",
              "pricing": "Pro: $49/mo. Enterprise: $199/mo. Annual: 20% off."}
        query = arguments.get("query", "").lower()
        for key, val in db.items():
            if key in query: return val
        return "No results found."
    
    elif name == "calculate":
        expr = arguments.get("expression", "")
        try: return str(eval(expr))
        except: return "Calculation error"
    
    elif name == "send_email":
        # In production: actually send. Here we simulate.
        return f"[SIMULATED] Email sent to {arguments['to']}"
    
    return "Unknown tool"


# ---------- THE FUNCTION CALLING LOOP ----------

def function_calling_agent(question):
    """Let the LLM decide which tools to call. We execute them."""
    
    messages = [{"role": "user", "content": question}]
    
    print(f"  User: {question}")
    
    # Step 1: Ask the LLM (it might want to call tools)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
        temperature=0,
    )
    
    msg = response.choices[0].message
    
    # Step 2: Did the LLM want to call any tools?
    if msg.tool_calls:
        print(f"  LLM wants to call {len(msg.tool_calls)} tool(s):")
        
        # Add the LLM's response to conversation
        messages.append(msg)
        
        # Execute each tool call
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            print(f"    🔧 {tc.function.name}({json.dumps(args)[:50]})")
            
            # YOUR CODE executes the tool (not the LLM!)
            result = execute_tool(tc.function.name, args)
            print(f"    📋 Result: {result[:60]}")
            
            # Send result back to the LLM
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result
            })
        
        # Step 3: LLM generates final answer using tool results
        final = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0,
        )
        answer = final.choices[0].message.content
    else:
        # LLM answered directly (no tools needed)
        answer = msg.content
    
    print(f"  Answer: {answer}")
    return answer


# ---------- TEST ----------

print("FUNCTION CALLING AGENT")
print("=" * 60)
print()

for q in [
    "What is the Pro plan price for a full year with annual discount?",
    "Send an email to alice@company.com about her refund status",
    "What is 2 + 2?",  # Might answer directly without tools!
]:
    function_calling_agent(q)
    print()

print("💡 The LLM DECIDES which tool. YOUR CODE executes. Safety boundary.")
print("💡 Multiple tools can be called in parallel (the LLM requests both at once).")
print("💡 Tool descriptions are CRITICAL. Bad description = wrong tool choice.")

---# 3️⃣ LangGraph State Machine (Production Agents)**Analogy:** A flowchart you can actually RUN. Boxes are functions. Arrows connect them. Some arrows are conditional: "if billing → go left, if technical → go right." This is how production agents work.**What we build:** Customer support router that classifies → routes → handles → responds.

In [ ]:
# ============================================================
# LANGGRAPH: A flowchart you can run
# classify → route → handle (billing/tech/general) → respond
# ============================================================

from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END

# ---------- STATE (the shared clipboard every node can read/write) ----------

class SupportState(TypedDict):
    query: str           # What the user asked
    category: str        # billing / technical / general
    context: str         # Retrieved information
    response: str        # Final response to user

# ---------- NODES (boxes in the flowchart) ----------

def classify_query(state: SupportState) -> SupportState:
    """Node 1: Figure out what type of question this is."""
    result, _ = ask(
        f"Classify this customer query into exactly one category: billing, technical, or general.\n"
        f"Query: {state['query']}\n"
        f"Reply with ONE word only: billing, technical, or general.",
    )
    state["category"] = result.strip().lower()
    print(f"    📋 Classified as: {state['category']}")
    return state

def handle_billing(state: SupportState) -> SupportState:
    """Node 2a: Handle billing questions."""
    state["context"] = "Pro: $49/mo. Enterprise: $199/mo. Annual: 20% off. Student: 50% off Pro."
    state["response"], _ = ask(
        f"You are a billing specialist. Answer using this info:\n{state['context']}\n\nQuestion: {state['query']}"
    )
    print(f"    💰 Billing handler responded")
    return state

def handle_technical(state: SupportState) -> SupportState:
    """Node 2b: Handle technical questions."""
    state["context"] = "Common fixes: restart app, clear cache, update to latest version. API docs at docs.example.com"
    state["response"], _ = ask(
        f"You are a tech support specialist. Answer using this info:\n{state['context']}\n\nQuestion: {state['query']}"
    )
    print(f"    🔧 Technical handler responded")
    return state

def handle_general(state: SupportState) -> SupportState:
    """Node 2c: Handle general questions."""
    state["response"], _ = ask(
        f"You are a friendly support agent. Answer briefly:\n{state['query']}"
    )
    print(f"    💬 General handler responded")
    return state

# ---------- ROUTING FUNCTION (the fork in the road) ----------

def route_by_category(state: SupportState) -> Literal["billing", "technical", "general"]:
    """Conditional edge: which handler should process this?"""
    cat = state.get("category", "general")
    if "bill" in cat: return "billing"
    if "tech" in cat: return "technical"
    return "general"

# ---------- BUILD THE GRAPH (the flowchart) ----------

graph = StateGraph(SupportState)

# Add nodes (boxes)
graph.add_node("classify", classify_query)
graph.add_node("billing", handle_billing)
graph.add_node("technical", handle_technical)
graph.add_node("general", handle_general)

# Set entry point
graph.set_entry_point("classify")

# Add conditional edges (the fork)
graph.add_conditional_edges("classify", route_by_category)

# All handlers lead to END
graph.add_edge("billing", END)
graph.add_edge("technical", END)
graph.add_edge("general", END)

# Compile the graph
app = graph.compile()

# ---------- TEST ----------

print("LANGGRAPH STATE MACHINE")
print("=" * 60)

test_queries = [
    "How much does the Enterprise plan cost per year?",
    "My API keeps returning 500 errors",
    "Hi, is anyone there? I just want to say thanks!",
]

for query in test_queries:
    print(f"\n  User: {query}")
    result = app.invoke({
        "query": query,
        "category": "",
        "context": "",
        "response": ""
    })
    print(f"  Response: {result['response'][:80]}...")

print("\n💡 Each query was classified, routed to the right handler, and answered.")
print("💡 This is a REAL production pattern. Add more nodes for more departments.")
print("💡 Add checkpointing to survive server restarts. Add HITL for approvals.")

---# 4️⃣ Multi-Agent Team (Researcher + Writer)**Analogy:** One person can't simultaneously research AND write AND edit well. A TEAM works better: researcher finds data, writer creates the draft, editor polishes. Each does ONE thing excellently.**What we build:** A 2-agent team using pure Python. Researcher gathers facts, writer creates content.

In [ ]:
# ============================================================
# MULTI-AGENT: Two specialists working together
# Agent 1 (Researcher): finds facts and data
# Agent 2 (Writer): turns research into a polished piece
# ============================================================

def researcher_agent(topic):
    """
    Agent 1: The Researcher
    Role: Find facts, numbers, and key points.
    Style: Analytical, cites data, no fluff.
    """
    
    system = """You are a Senior Research Analyst with 10 years of experience.
Your job: find 5 KEY FACTS about the topic. Each fact must include a specific 
number or data point. Be concise. Bullet points only. No opinions."""
    
    research, tokens = ask(f"Research this topic: {topic}", system=system)
    return research, tokens


def writer_agent(research, format_type="blog"):
    """
    Agent 2: The Writer
    Role: Turn research into engaging content.
    Style: Clear, engaging, uses the researcher's data.
    """
    
    system = """You are an award-winning tech writer.
Your job: turn research notes into a short, engaging blog post (150 words max).
Rules: use ALL the data from the research. Make it interesting. Include a hook.
Do NOT add facts that aren't in the research."""
    
    article, tokens = ask(
        f"Write a {format_type} post from this research:\n{research}",
        system=system
    )
    return article, tokens


# ---------- ORCHESTRATOR (the project manager) ----------

def multi_agent_pipeline(topic):
    """
    The Orchestrator coordinates the team:
    Step 1: Researcher finds facts
    Step 2: Writer creates content from those facts
    """
    
    print(f"  Topic: {topic}")
    print(f"  {'─' * 50}")
    
    # Step 1: Research
    print("  🔬 Researcher working...")
    start = time.time()
    research, t1 = researcher_agent(topic)
    print(f"     Done in {time.time()-start:.1f}s ({t1} tokens)")
    print(f"     Research: {research[:100]}...")
    
    # Step 2: Write
    print("  ✍️  Writer working...")
    start = time.time()
    article, t2 = writer_agent(research)
    print(f"     Done in {time.time()-start:.1f}s ({t2} tokens)")
    
    print(f"\n  📝 FINAL ARTICLE:")
    print(f"  {'─' * 50}")
    print(f"  {article}")
    print(f"\n  Total cost: {t1 + t2} tokens")
    
    return article


# ---------- RUN ----------

print("MULTI-AGENT TEAM")
print("=" * 60)
print()

multi_agent_pipeline("Electric vehicles in 2025")

print("\n💡 Researcher found facts. Writer turned them into content.")
print("💡 Each agent has a specialized system prompt (role + rules).")
print("💡 2-3 agents is the sweet spot. More agents = more cost + coordination overhead.")

---# 5️⃣ Agent Memory (Remember Past Conversations)**Analogy:** Talking to someone with amnesia — every message is the first message. Memory fixes this. Short-term: last 5 messages. Entity memory: auto-extract facts about people. Both together: personalized conversations.**What we build:** A chatbot with sliding window memory + automatic entity extraction.

In [ ]:
# ============================================================
# MEMORY: Short-term (recent messages) + Entity (facts about people)
# Without memory: every message is the first message
# With memory: the agent remembers your name, preferences, history
# ============================================================

class ChatMemory:
    """
    Two types of memory:
    1. Short-term: last N messages (like working memory)
    2. Entity: facts about people/things (like a personal CRM)
    """
    
    def __init__(self, max_messages=6):
        self.messages = []          # Short-term: recent conversation
        self.max_messages = max_messages
        self.entities = {}          # Entity: facts about people/things
    
    def add_message(self, role, content):
        """Add a message. Drop oldest if over limit."""
        self.messages.append({"role": role, "content": content})
        
        # Keep only the last N messages (sliding window)
        if len(self.messages) > self.max_messages:
            dropped = self.messages.pop(0)
            print(f"    💭 Forgot oldest message: '{dropped['content'][:30]}...'")
    
    def extract_entities(self, text):
        """Auto-extract facts about people from the conversation."""
        result, _ = ask(
            f"Extract any facts about people from this text. "
            f"Return as JSON: {{'person_name': {{'fact_type': 'value'}}}}. "
            f"If no facts about people, return {{}}. JSON only.\n"
            f"Text: {text}"
        )
        try:
            clean = result.replace("```json","").replace("```","").strip()
            entities = json.loads(clean)
            for person, facts in entities.items():
                if person not in self.entities:
                    self.entities[person] = {}
                self.entities[person].update(facts)
                print(f"    🧠 Remembered: {person} → {facts}")
        except:
            pass
    
    def get_context(self):
        """Build context string from memory."""
        context = ""
        if self.entities:
            context += "Known facts about people:\n"
            for person, facts in self.entities.items():
                context += f"  {person}: {json.dumps(facts)}\n"
            context += "\n"
        return context
    
    def chat(self, user_message):
        """Full chat with memory."""
        # Extract entities from user message
        self.extract_entities(user_message)
        
        # Add user message to history
        self.add_message("user", user_message)
        
        # Build system prompt with entity memory
        entity_context = self.get_context()
        system = f"""You are a helpful assistant. Be concise.
        
{entity_context}
Use the known facts to personalize your responses."""
        
        # Call LLM with recent message history
        messages = [{"role": "system", "content": system}] + self.messages
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0
        )
        
        answer = response.choices[0].message.content.strip()
        self.add_message("assistant", answer)
        
        return answer


# ---------- TEST ----------

print("AGENT MEMORY")
print("=" * 60)

memory = ChatMemory(max_messages=6)

conversation = [
    "Hi! I'm Alice and I work at TechCorp as a VP of Engineering.",
    "I prefer Python over JavaScript for backend work.",
    "What programming language would you recommend for my team?",
    "We're building a data pipeline. Any framework suggestions?",
    "Remind me — what's my role again?",  # Tests entity memory!
]

for msg in conversation:
    print(f"\n  You: {msg}")
    response = memory.chat(msg)
    print(f"  Bot: {response[:100]}...")

print(f"\n\n  Entity Memory (auto-extracted):")
for person, facts in memory.entities.items():
    print(f"    {person}: {json.dumps(facts)}")

print(f"\n  Message History: {len(memory.messages)} messages (max {memory.max_messages})")
print(f"\n💡 The bot remembered Alice's name, company, and role WITHOUT being told to.")
print(f"💡 Entity extraction happens automatically on every message.")
print(f"💡 Old messages drop off, but entity facts persist forever.")

---# 6️⃣ Human-in-the-Loop (Approval Gates)**Analogy:** A self-driving car that handles highways but STOPS at confusing intersections and asks the human driver to take over. Safe actions run automatically. Risky actions need human approval.**What we build:** Risk-tiered approval system. Search = auto. Refund = needs approval. Delete = permanently blocked.

In [ ]:
# ============================================================
# HUMAN-IN-THE-LOOP: Risk-tiered approval gates
# AUTO: search, calculate, read (safe actions)
# APPROVAL: refund, send email, modify account (risky)
# BLOCKED: delete account, admin actions (dangerous)
# ============================================================

class ApprovalGate:
    """Classify every agent action by risk level."""
    
    RISK_TIERS = {
        # AUTO — safe, no approval needed
        "search_database": "auto",
        "calculate": "auto",
        "get_date": "auto",
        "read_account": "auto",
        
        # APPROVAL — needs human to say "yes"
        "process_refund": "approval",
        "send_email": "approval",
        "modify_account": "approval",
        "change_plan": "approval",
        
        # BLOCKED — never allowed
        "delete_account": "blocked",
        "access_admin": "blocked",
        "export_all_data": "blocked",
    }
    
    def __init__(self):
        self.pending_approvals = []
        self.audit_log = []
    
    def check(self, action, details):
        """Check if an action is allowed, needs approval, or is blocked."""
        
        risk = self.RISK_TIERS.get(action, "approval")  # Unknown = needs approval
        
        self.audit_log.append({
            "action": action,
            "risk": risk,
            "details": str(details)[:100],
        })
        
        if risk == "auto":
            print(f"    ✅ AUTO-APPROVED: {action}")
            return True, "auto"
        
        elif risk == "blocked":
            print(f"    🚫 PERMANENTLY BLOCKED: {action}")
            return False, "blocked"
        
        else:  # approval needed
            print(f"    ⏸️  NEEDS APPROVAL: {action} — {json.dumps(details)[:60]}")
            # In production: send notification, wait for human response
            # Here we simulate automatic approval for demo
            approved = True  # Simulated: human clicks "approve"
            status = "approved" if approved else "denied"
            print(f"    {'✅' if approved else '❌'} Human {status}")
            return approved, "approval"
    
    def report(self):
        print(f"\n  AUDIT LOG ({len(self.audit_log)} actions):")
        for entry in self.audit_log:
            emoji = {"auto":"✅","approval":"⏸️","blocked":"🚫"}[entry["risk"]]
            print(f"    {emoji} {entry['action']:20s} [{entry['risk']:8s}]")


# ---------- SIMULATE AGENT ACTIONS ----------

print("HUMAN-IN-THE-LOOP APPROVAL GATES")
print("=" * 60)
print()

gate = ApprovalGate()

# Simulate a sequence of agent actions
agent_actions = [
    ("search_database",  {"query": "refund policy"}),
    ("calculate",        {"expression": "49 * 12 * 0.8"}),
    ("process_refund",   {"amount": 499, "order": "ORD-123", "reason": "defective"}),
    ("send_email",       {"to": "customer@mail.com", "subject": "Refund processed"}),
    ("delete_account",   {"user_id": "USR-456"}),
    ("read_account",     {"user_id": "USR-456"}),
    ("access_admin",     {"action": "view_all_users"}),
]

for action, details in agent_actions:
    allowed, risk_level = gate.check(action, details)
    if allowed:
        print(f"    → Executing {action}...")
    else:
        print(f"    → Skipped (blocked)")
    print()

gate.report()

print("\n💡 Safe actions auto-run. Risky actions pause for human approval.")
print("💡 Dangerous actions are PERMANENTLY blocked. No override.")
print("💡 Full audit log for compliance. Every action tracked.")

---# 7️⃣ Safe Code Execution (Sandbox)**Analogy:** Giving a child a chemistry set. You would NOT let them experiment in your kitchen. You set up a separate table with protective covers. If they spill something, no damage to your home.**What we build:** LLM writes Python code. We execute it in a restricted environment. See the output.

In [ ]:
# ============================================================
# CODE EXECUTION: LLM writes code, we run it safely
# In production: use E2B sandbox ($0.10/hr, fully isolated)
# Here: we use restricted exec() for demo
# ============================================================

import io, sys

def safe_execute(code_string, timeout_hint=5):
    """
    Run Python code in a restricted way.
    Captures print output. Catches errors.
    
    WARNING: This is a DEMO. In production, use E2B sandbox
    which runs code in an isolated cloud container.
    """
    
    # Capture stdout
    old_stdout = sys.stdout
    sys.stdout = captured = io.StringIO()
    
    try:
        # Execute the code
        exec(code_string, {"__builtins__": __builtins__}, {})
        output = captured.getvalue()
        return output.strip(), None
    except Exception as e:
        return captured.getvalue(), str(e)
    finally:
        sys.stdout = old_stdout


def code_agent(task, max_retries=3):
    """
    LLM writes code to solve a task.
    If it fails, LLM sees the error and fixes it.
    Up to 3 retries (like a developer debugging).
    """
    
    print(f"  Task: {task}")
    
    for attempt in range(1, max_retries + 1):
        # Ask LLM to write code
        if attempt == 1:
            prompt = f"Write Python code to: {task}\nUse print() to show results. Code only, no markdown."
        else:
            prompt = f"""The previous code had an error: {error}

Fix the code. Write the COMPLETE corrected code. Code only, no markdown.
Task: {task}"""
        
        code_text, _ = ask(prompt)
        
        # Clean markdown fences if present
        code_text = code_text.replace("```python", "").replace("```", "").strip()
        
        print(f"\n  Attempt {attempt}:")
        print(f"  Code: {code_text[:80]}...")
        
        # Execute
        output, error = safe_execute(code_text)
        
        if error is None:
            print(f"  ✅ Output: {output[:100]}")
            return output
        else:
            print(f"  ❌ Error: {error}")
            if attempt < max_retries:
                print(f"  🔄 LLM will read the error and fix the code...")
    
    return "Failed after 3 attempts."


# ---------- TEST ----------

print("CODE EXECUTION AGENT")
print("=" * 60)
print()

tasks = [
    "Calculate the first 10 Fibonacci numbers and print them as a list",
    "Find all prime numbers between 1 and 50 and print the count and the list",
    "Calculate compound interest: $1000 principal, 5% rate, 10 years. Print each year's balance.",
]

for task in tasks:
    code_agent(task)
    print("\n" + "─" * 60 + "\n")

print("💡 The LLM wrote code, we ran it, and saw the output.")
print("💡 If code had an error, the LLM read the error message and FIXED it.")
print("💡 In production: use E2B sandbox (pip install e2b-code-interpreter).")
print("💡 NEVER run LLM-generated code on your real servers without a sandbox.")

---# 🏁 Summary — Agent Architecture Decision Guide| Pattern | Use When | Cost | Complexity ||---------|----------|------|------------|| **ReAct** | Simple tool use (80% of cases) | $ | Low || **Function Calling** | Need structured tool invocation | $ | Low || **LangGraph** | Routing, loops, state, HITL | $$ | Medium || **Multi-Agent** | Tasks need different expertise | $$$ | Medium || **Plan-Execute** | 5+ coordinated steps | $$ | Medium || **Reflexion** | Quality-critical (code, writing) | $$$ | High |**The ladder:** ReAct → Function Calling → LangGraph → Multi-Agent. Use the simplest that works.**Non-negotiables for production:**- Max iteration limit (prevent infinite loops)- Human-in-the-loop for risky actions- Full audit logging of every action- Tool descriptions are your #1 quality lever